# Pilot fall-risk classification from lab walking data

This notebook compares the planned predictor sets using participant-level fall-risk labels: Xsens gait features, generic LIMU-BERT-X, stroke-adapted LIMU-BERT-X, generic HART, stroke-adapted HART, and gait features combined with each stroke-adapted embedding. Clinical questionnaire and test scores are not predictors because they contribute to the fall-risk classification label.

NW trial embeddings and gait features are averaged within each participant, giving one sample per labelled participant. Evaluation uses nested leave-one-participant-out cross-validation (LOPO). The inner folds select the logistic-regression regularization strength. Scaling and tuning are fitted only on training participants. With 16 labelled participants, this is a pilot analysis, not a clinical performance estimate.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             recall_score, roc_auc_score)
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

search_paths=[Path.cwd(),*Path.cwd().parents]
repo_root=next((p for p in search_paths if (p/'Data/analysis/master_analysis_table_pca.csv').exists()),None)
if repo_root is None:
    raise FileNotFoundError('Could not find Data/analysis/master_analysis_table_pca.csv from the current directory.')
master=pd.read_csv(repo_root/'Data/analysis/master_analysis_table_pca.csv')
results_dir=repo_root/'Data/analysis/fall_risk_results'
results_dir.mkdir(parents=True,exist_ok=True)
print(f'Loaded {len(master)} NW trial rows from {master.participant_id.nunique()} participants.')

In [ ]:
label_col='fall_risk_label'
labels_by_participant=master.groupby('participant_id')[label_col].first().dropna()
label_counts=labels_by_participant.value_counts()
print(f'Labelled participants: {len(labels_by_participant)}')
print(label_counts.to_string())

# Aggregate repeated NW trials to one row per participant.
gait_columns=[
    'sacrum_walk_step_count',
    'sacrum_walk_cadence_steps_min',
    'sacrum_walk_mean_step_time_s',
    'sacrum_walk_duration_s',
]
participant_gait=master.groupby('participant_id')[gait_columns].mean()
embedding_columns={
    'LIMU-BERT-X generic':[c for c in master if c.startswith('limu_generic_embedding_')],
    'LIMU-BERT-X stroke-adapted':[c for c in master if c.startswith('limu_stroke_adapted_embedding_')],
    'HART generic':[c for c in master if c.startswith('hart_generic_embedding_')],
    'HART stroke-adapted':[c for c in master if c.startswith('hart_stroke_adapted_embedding_')],
}
participant_embeddings={
    name:master.groupby('participant_id')[cols].mean()
    for name,cols in embedding_columns.items()
}
participant_gait=participant_gait.loc[labels_by_participant.index]
participant_embeddings={name:frame.loc[labels_by_participant.index] for name,frame in participant_embeddings.items()}
y=labels_by_participant.astype(str)
print(f'Predictor sample size: {len(y)} participants')

In [ ]:
# Separate predictor sets, matching the planned comparisons.
feature_sets={
    'Xsens gait features':participant_gait,
    'LIMU-BERT-X generic':participant_embeddings['LIMU-BERT-X generic'],
    'LIMU-BERT-X stroke-adapted':participant_embeddings['LIMU-BERT-X stroke-adapted'],
    'HART generic':participant_embeddings['HART generic'],
    'HART stroke-adapted':participant_embeddings['HART stroke-adapted'],
    'Gait + LIMU-BERT-X stroke-adapted':pd.concat([participant_gait,participant_embeddings['LIMU-BERT-X stroke-adapted']],axis=1),
    'Gait + HART stroke-adapted':pd.concat([participant_gait,participant_embeddings['HART stroke-adapted']],axis=1),
}
for name,frame in feature_sets.items():
    print(f'{name}: {frame.shape[1]} predictor variables')

## Nested leave-one-participant-out evaluation

Each outer fold holds one participant out for evaluation. Within the remaining participants, three-fold stratified cross-validation selects the regularization parameter `C`. The pipeline standardizes the inputs inside each fold and fits L2-regularized logistic regression. The majority-class prediction from the training fold is included as a baseline.

In [ ]:
C_values=np.logspace(-8,4,25)
outer_cv=LeaveOneOut()
inner_cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=42)
positive_label='Faller'
all_oof=[]
summary_rows=[]

for set_name,features in feature_sets.items():
    X=features.to_numpy(dtype=float)
    y_array=y.to_numpy()
    participant_ids=features.index.to_numpy()
    oof_pred=np.empty(len(y_array),dtype=object)
    oof_prob=np.empty(len(y_array),dtype=float)
    baseline_pred=np.empty(len(y_array),dtype=object)
    selected_C=[]
    for train_idx,test_idx in outer_cv.split(X,y_array):
        pipeline=Pipeline([
            ('scale',StandardScaler()),
            ('logistic',LogisticRegression(penalty='l2',solver='liblinear',max_iter=1000)),
        ])
        search=GridSearchCV(
            pipeline,
            param_grid={'logistic__C':C_values},
            scoring='balanced_accuracy',
            cv=inner_cv,
            n_jobs=1,
        )
        search.fit(X[train_idx],y_array[train_idx])
        model=search.best_estimator_
        oof_pred[test_idx]=model.predict(X[test_idx])
        positive_index=list(model.classes_).index(positive_label)
        oof_prob[test_idx]=model.predict_proba(X[test_idx])[0,positive_index]
        baseline_pred[test_idx]=pd.Series(y_array[train_idx]).mode().iloc[0]
        selected_C.append(search.best_params_['logistic__C'])
        all_oof.append({
            'predictor_set':set_name,
            'participant_id':participant_ids[test_idx[0]],
            'actual_label':y_array[test_idx[0]],
            'predicted_label':oof_pred[test_idx[0]],
            'faller_probability':oof_prob[test_idx[0]],
            'majority_baseline_prediction':baseline_pred[test_idx[0]],
            'selected_C':search.best_params_['logistic__C'],
        })
    tn,fp,fn,tp=confusion_matrix(y_array,oof_pred,labels=['Non-Faller','Faller']).ravel()
    summary_rows.append({
        'predictor_set':set_name,
        'n_participants':len(y_array),
        'n_features':X.shape[1],
        'LOPO_accuracy':accuracy_score(y_array,oof_pred),
        'LOPO_balanced_accuracy':balanced_accuracy_score(y_array,oof_pred),
        'LOPO_sensitivity_faller':recall_score(y_array,oof_pred,pos_label='Faller'),
        'LOPO_specificity_nonfaller':recall_score(y_array,oof_pred,pos_label='Non-Faller'),
        'LOPO_ROC_AUC':roc_auc_score((y_array==positive_label).astype(int),oof_prob),
        'majority_baseline_accuracy':accuracy_score(y_array,baseline_pred),
        'majority_baseline_balanced_accuracy':balanced_accuracy_score(y_array,baseline_pred),
        'median_selected_C':float(np.median(selected_C)),
        'fraction_folds_at_min_C':float(np.mean(np.isclose(selected_C,C_values[0]))),
        'fraction_folds_at_max_C':float(np.mean(np.isclose(selected_C,C_values[-1]))),
        'true_negatives':tn,'false_positives':fp,'false_negatives':fn,'true_positives':tp,
    })

summary=pd.DataFrame(summary_rows)
oof_predictions=pd.DataFrame(all_oof)
summary.to_csv(results_dir/'fall_risk_lopo_summary.csv',index=False)
oof_predictions.to_csv(results_dir/'fall_risk_lopo_predictions.csv',index=False)
display(summary.round(3))

## Model comparison plots

Balanced accuracy and ROC AUC summarize the outer held-out predictions. The dashed line at 0.5 is the no-discrimination reference for these metrics. With 16 people, small changes in the predictions can shift the scores substantially.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
for ax,metric,title in [
    (axes[0],'LOPO_balanced_accuracy','Leave-one-participant-out balanced accuracy'),
    (axes[1],'LOPO_ROC_AUC','Leave-one-participant-out ROC AUC'),
]:
    ordered=summary.sort_values(metric)
    ax.barh(ordered.predictor_set,ordered[metric],color='#4778a8')
    ax.axvline(.5,color='black',linestyle='--',linewidth=1)
    ax.set(xlim=(0,1),xlabel=metric.replace('LOPO_',''),title=title)
    ax.grid(axis='x',alpha=.2)
fig.tight_layout()
fig.savefig(results_dir/'fall_risk_model_comparison.png',dpi=160)
plt.show()

## Interpretation cautions

- Fall-risk labels are participant-level labels from the classification file. Clinical scores used to create those labels are not model inputs.
- All NW trials from a participant are averaged before modeling; the outer fold tests a participant not used for training or tuning.
- A majority-class baseline is included because the labelled sample is small (9 fallers, 7 non-fallers).
- This is a pilot with only 16 labelled participants. Report the confusion counts and metrics as exploratory; they are not reliable clinical performance estimates. Check whether the selected regularization strength repeatedly hits the search-grid boundary.
- The gait measures are related and may carry overlapping information. The combined models test whether adding an adapted embedding improves this small-sample comparison, but results require validation in a larger independent cohort.